# Dataset Preperation

In [4]:
import os
import cv2
import gc
import json
import numpy as np
import pandas as pd
import mediapipe as mp

# FACE SEGMENTATION ENGINE
def segment_face(image, face_mesh):
    """
    Accepts loaded image, returns cropped face with black background.
    """
    h_img, w_img, _ = image.shape
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_image)

    if not results.multi_face_landmarks:
        return None 

    landmarks = results.multi_face_landmarks[0]
    
    # Get Geometry
    points = np.array([(int(lm.x * w_img), int(lm.y * h_img)) for lm in landmarks.landmark])
    x, y, w_box, h_box = cv2.boundingRect(points)
    
    # Filter: Too Small or Weird Aspect Ratio
    if w_box < 50 or h_box < 50: return None
    aspect = h_box / w_box
    if aspect < 0.6 or aspect > 2.5: return None

    # Masking
    mask = np.zeros((h_img, w_img), dtype=np.uint8)
    hull = cv2.convexHull(points)
    cv2.fillConvexPoly(mask, hull, 255)
    segmented = cv2.bitwise_and(image, image, mask=mask)

    # Cropping with Padding
    pad = 10
    x = max(0, x - pad)
    y = max(0, y - pad)
    w_box = min(w_img, x + w_box + 2*pad) - x
    h_box = min(h_img, y + h_box + 2*pad) - y
    
    return segmented[y:y+h_box, x:x+w_box]

## Casual Conversation v2

Download the dataset from [here](https://ai.meta.com/datasets/casual-conversations-v2-downloads/). This should provide you with the following .zip files: 

- CCv2_frames_part_1.zip
- CCv2_frames_part_2.zip
- CCv2_frames_part_3.zip
- CCv2_frames_part_4.zip
- CCv2_frames_part_5.zip
- CCv2_annotations.zip

### Process Casual Conversation v2 Dataset

In [ ]:
import os
import json
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm
import mediapipe as mp
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# CONFIG
# ============================================================

CCV2_ROOT = r"CasualConversationv2_Dataset\Images"
CCV2_JSON = r"CasualConversationv2_Dataset\Annotations\CasualConversationsV2.json"
CCV2_OUTPUT_DIR = r"CasualConversationv2_Dataset\Segmented_CCV2"

confidence_filter = None        # None | ["low"] | ["medium"] | ["high"]
NUM_WORKERS = 4                 # number of FaceMesh instances
BATCH_SIZE = 32                 # images per scheduling batch

os.makedirs(CCV2_OUTPUT_DIR, exist_ok=True)

# ============================================================
# PARSERS
# ============================================================

def parse_mst_label(mst_dict):
    scale_str = mst_dict.get("scale", "")
    digits = ''.join(c for c in scale_str if c.isdigit())
    return int(digits) if digits.isdigit() else None


def parse_gender(gender_raw):
    if not gender_raw:
        return None
    g = gender_raw.strip().lower()
    if "woman" in g or "female" in g:
        return "female"
    elif "man" in g or "male" in g:
        return "male"
    return None


# ============================================================
# UTIL: chunk list into batches
# ============================================================

def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


# ============================================================
# WORKER: one FaceMesh per thread
# ============================================================

def process_batch(batch):
    records = []

    with mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as face_mesh:

        for subject_id, fname, in_path, out_path, mst_label, gender in batch:

            img = cv2.imread(in_path)
            if img is None:
                continue

            crop = segment_face(img, face_mesh)
            if crop is None:
                continue

            cv2.imwrite(out_path, crop)

            records.append({
                "filename": os.path.join(subject_id, fname).replace("\\", "/"),
                "original_image": fname,
                "cropped_image": fname,
                "subject_id": subject_id,
                "mst_label": mst_label,
                "gender": gender
            })

    return records


# ============================================================
# MAIN PIPELINE
# ============================================================

def process_casual_conversations_v2(
    confidence_filter=None,
    num_workers=4,
    batch_size=32
):
    print("\n--- Processing Casual Conversations V2 Dataset (batched + parallel) ---")

    # --------------------------------------------------------
    # Load + deduplicate subjects
    # --------------------------------------------------------
    with open(CCV2_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    subjects = {}

    for entry in data:
        sid = entry["subject_id"]
        if sid in subjects:
            continue

        mst = entry.get("monk_skin_tone", {})
        mst_label = parse_mst_label(mst)
        if mst_label is None:
            continue

        conf = mst.get("confidence", "").lower()
        if confidence_filter and conf not in confidence_filter:
            continue

        gender = parse_gender(entry.get("gender", ""))

        subjects[sid] = (mst_label, gender)

    print(f"Loaded {len(subjects)} unique subjects.")

    # --------------------------------------------------------
    # Build flat image task list
    # --------------------------------------------------------
    tasks = []

    for subject_id, (mst_label, gender) in subjects.items():
        subj_in = os.path.join(CCV2_ROOT, subject_id)
        if not os.path.isdir(subj_in):
            continue

        subj_out = os.path.join(CCV2_OUTPUT_DIR, subject_id)
        os.makedirs(subj_out, exist_ok=True)

        for fname in os.listdir(subj_in):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            in_path = os.path.join(subj_in, fname)
            out_path = os.path.join(subj_out, fname)

            tasks.append(
                (subject_id, fname, in_path, out_path, mst_label, gender)
            )

    print(f"Queued {len(tasks)} images.")

    # --------------------------------------------------------
    # Create batches
    # --------------------------------------------------------
    batches = list(chunked(tasks, batch_size))
    print(f"Processing {len(batches)} batches (batch_size={batch_size})")

    # --------------------------------------------------------
    # Parallel execution
    # --------------------------------------------------------
    all_records = []

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [
            executor.submit(process_batch, batch)
            for batch in batches
        ]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Processing batches"
        ):
            all_records.extend(future.result())

    # --------------------------------------------------------
    # Build CSV
    # --------------------------------------------------------
    df_out = pd.DataFrame(all_records)

    # Drop non-binary / undefined genders
    df_out = df_out.dropna(subset=["gender"])

    out_csv = os.path.join(CCV2_OUTPUT_DIR, "annotations.csv")
    df_out.to_csv(out_csv, index=False)

    print(f"Saved {len(df_out)} rows → {out_csv}")
    return out_csv


# ============================================================
# RUN
# ============================================================

ccv2_out_csv_path = process_casual_conversations_v2(
    confidence_filter=confidence_filter,
    num_workers=NUM_WORKERS,
    batch_size=BATCH_SIZE
)



--- Processing Casual Conversations V2 Dataset (batched + parallel) ---
Loaded 5567 unique subjects.
Queued 1000 images.
Processing 32 batches (batch_size=32)


Processing batches: 100%|██████████| 32/32 [00:07<00:00,  4.11it/s]

Saved 859 rows → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2_test\annotations.csv


## Monk Skin Tone

Download the dataset from [here](https://skintone.google/mste-dataset). This should provide you with the following .zip file: 

- mst-e_data.zip

### Process Monk Skin Tone Dataset

In [ ]:
import os
import gc
import cv2
import pandas as pd
from tqdm import tqdm
import mediapipe as mp

# ============================================================
# CONFIG
# ============================================================

MST_E_ROOT = r'DatasetAnnotation\MonkSkinToneDataset\mst-e_data'
MST_E_CSV_RAW = os.path.join(MST_E_ROOT, 'mst-e_image_details.csv')
MST_E_OUTPUT_DIR = r'DatasetAnnotation\MonkSkinToneDataset\Segmented_MSTE'

os.makedirs(MST_E_OUTPUT_DIR, exist_ok=True)


# ============================================================
# PROCESS MST-E
# ============================================================

def process_mste():
    print("\n--- Processing MST-E Dataset ---")

    df = pd.read_csv(MST_E_CSV_RAW)

    # Standardise column names
    df = df.rename(columns={"MST": "mst_label"})

    records = []

    with mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as face_mesh:

        for i, row in tqdm(
            enumerate(df.itertuples(index=False)),
            total=len(df),
            desc="Segmenting MST-E faces"
        ):
            subject_id = str(row.subject_name)
            fname = str(row.image_ID)

            in_path = os.path.join(MST_E_ROOT, subject_id, fname)
            out_dir = os.path.join(MST_E_OUTPUT_DIR, subject_id)
            os.makedirs(out_dir, exist_ok=True)

            out_path = os.path.join(out_dir, fname)

            if not os.path.exists(in_path):
                continue

            img = cv2.imread(in_path)
            if img is None:
                continue

            crop = segment_face(img, face_mesh)
            if crop is None:
                continue

            cv2.imwrite(out_path, crop)

            records.append({
                "filename": os.path.join(subject_id, fname).replace("\\", "/"),
                "mst_label": int(row.mst_label),
                "subject_id": subject_id
            })

            if i % 200 == 0:
                gc.collect()

    df_out = pd.DataFrame(records)

    out_csv = os.path.join(MST_E_OUTPUT_DIR, "annotations.csv")
    df_out.to_csv(out_csv, index=False)

    print(f"MST-E Done. Saved {len(df_out)} images.")
    print(f"CSV written to: {out_csv}")

    return out_csv


# ============================================================
# RUN
# ============================================================

mste_out_csv_path = process_mste()



--- Processing MST-E Dataset ---


Segmenting MST-E faces: 100%|██████████| 1546/1546 [02:39<00:00,  9.69it/s]

MST-E Done. Saved 1388 images.
CSV written to: G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE_2\annotations.csv


## Process FACET Dataset

Download the dataset from [here](https://ai.meta.com/datasets/facet-downloads/). This should provide you with the following files: 

- annotations.tar.gz
- imgs_1.tar.gz
- imgs_2.tar.gz
- imgs_3.tar.gz

Extract the files ensuring all the images are stored in a singular directory.

In [ ]:
import os
import gc
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import mediapipe as mp

# ============================================================
# CONFIG
# ============================================================

FACET_ROOT_IMAGES = r'FACET_Dataset\Images'
FACET_CSV_RAW = r'FACET_Dataset\Annotations\annotations\annotations.csv'

consensus_threshold = 0.2

FACET_OUTPUT_DIR = rf'FACET_Dataset\Segmented_FACET_{consensus_threshold}_fixed_label'
os.makedirs(FACET_OUTPUT_DIR, exist_ok=True)

# ============================================================
# PROCESS FACET
# ============================================================

def parse_facet_gender(row):
    """
    Map FACET gender presentation columns to binary gender.

    Rules:
    - masc > fem  -> male
    - fem > masc  -> female
    - ties, non-binary, NA -> None
    """

    masc = getattr(row, "gender_presentation_masc", 0)
    fem = getattr(row, "gender_presentation_fem", 0)

    try:
        masc = float(masc)
        fem = float(fem)
    except Exception:
        return None

    if masc > fem:
        return "male"
    elif fem > masc:
        return "female"

    return None

def process_facet(consensus_threshold=0.5, fixed_label=False):
    print("\n--- Processing FACET Dataset ---")

    # --------------------------------------------------------
    # Load annotations
    # --------------------------------------------------------
    df = pd.read_csv(FACET_CSV_RAW)

    skin_cols = [c for c in df.columns if c.startswith("skin_tone_")]

    # Optional gender column (FACET-dependent)
    gender_col = None
    for gcol in ["gender", "sex"]:
        if gcol in df.columns:
            gender_col = gcol
            break

    # Must have at least one vote
    df = df[df[skin_cols].sum(axis=1) > 0]

    # --------------------------------------------------------
    # Consensus agreement filtering
    # --------------------------------------------------------
    max_votes = df[skin_cols].max(axis=1)
    total_votes = df[skin_cols].sum(axis=1)
    agreement_ratio = max_votes / total_votes

    initial_len = len(df)
    df = df[agreement_ratio >= consensus_threshold]

    print(
        f"Consensus Filter: Dropped {initial_len - len(df)} images "
        f"with agreement < {consensus_threshold:.2f}"
    )

    # --------------------------------------------------------
    # MST label assignment
    # --------------------------------------------------------
    if fixed_label:
        df["mst_label"] = (
            df[skin_cols]
            .idxmax(axis=1)
            .str.replace("skin_tone_", "", regex=False)
        )
        df = df[df["mst_label"] != "na"]
        df["mst_label"] = df["mst_label"].astype(int)
    else:
        class_indices = np.arange(1, len(skin_cols) + 1)
        df["mst_label"] = (
            df[skin_cols]
            .mul(class_indices)
            .sum(axis=1)
            / df[skin_cols].sum(axis=1)
        )
        df = df.dropna(subset=["mst_label"])

    # --------------------------------------------------------
    # Face segmentation
    # --------------------------------------------------------
    records = []

    with mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as face_mesh:

        for i, row in tqdm(
            enumerate(df.itertuples(index=False)),
            total=len(df),
            desc="Segmenting faces"
        ):
            fname = row.filename
            input_path = os.path.join(FACET_ROOT_IMAGES, fname)
            output_path = os.path.join(FACET_OUTPUT_DIR, fname)

            if not os.path.exists(input_path):
                continue

            img = cv2.imread(input_path)
            if img is None:
                continue

            crop = segment_face(img, face_mesh)
            if crop is None:
                continue

            cv2.imwrite(output_path, crop)

            gender_val = parse_facet_gender(row)

            records.append({
                "filename": fname,                 # evaluator compatibility
                "original_image": fname,           # CCV2-style
                "cropped_image": fname,            # CCV2-style
                "mst_label": row.mst_label,
                "subject_id": os.path.splitext(fname)[0],
                "gender": gender_val
            })

            if i % 500 == 0:
                gc.collect()

    df_out = pd.DataFrame(records)
    
    # Removing entries for non_binary & na
    df_out = df_out.dropna(subset=["gender"])

    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------
    out_csv = os.path.join(FACET_OUTPUT_DIR, "annotations.csv")
    df_out.to_csv(out_csv, index=False)

    print(f"FACET Done. Saved {len(df_out)} images.")
    print(f"CSV written to: {out_csv}")

    return out_csv


# ============================================================
# RUN
# ============================================================

facet_out_csv_path = process_facet(
    consensus_threshold=consensus_threshold,
    fixed_label=True
)


--- Processing FACET Dataset ---
Consensus Filter: Dropped 189 images with agreement < 0.20


Segmenting faces: 100%|██████████| 40211/40211 [22:52<00:00, 29.29it/s]

FACET Done. Saved 2677 images.
CSV written to: G:\Thesis\FACET_Dataset\TESTING_2_Segmented_FACET_0.2_continuous\annotations.csv


### Convert MST_LABEL -> 3MST Bucket

In [8]:
import pandas as pd


def rebin_labels_1to3(
    input_csv: str,
    output_csv: str,
    label_column: str = "label"
):
    """
    Reads a CSV, converts label values from 1–10 to 1–3, and saves a new CSV.

    Binning logic:
        1–3   -> 1
        4–7   -> 2
        8–10  -> 3

    Args:
        input_csv (str): Path to input CSV
        output_csv (str): Path to output CSV
        label_column (str): Name of label column (default: 'label')
    """

    def to_bin(x):
        if pd.isna(x):
            return None
        x = int(x)
        if 1 <= x <= 3:
            return 1
        elif 4 <= x <= 7:
            return 2
        else:
            return 3

    df = pd.read_csv(input_csv)

    if label_column not in df.columns:
        raise ValueError(
            f"Column '{label_column}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    df[label_column] = df[label_column].apply(to_bin)

    df.to_csv(output_csv, index=False)
    print(f"Saved re-binned CSV → {output_csv}")


In [ ]:
rebin_labels_1to3(
    input_csv = r"FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv",
    output_csv = r"FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations_3mst.csv",
    label_column = "mst_label"
)

Saved re-binned CSV → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations_3mst.csv


In [ ]:
rebin_labels_1to3(
    input_csv = r"MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv",
    output_csv = r"MonkSkinTone_Dataset\Segmented_MSTE\annotations_3mst.csv",
    label_column = "mst_label"
)

Saved re-binned CSV → G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations_3mst.csv


In [ ]:
rebin_labels_1to3(
    input_csv = r"CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv",
    output_csv = r"CasualConversationv2_Dataset\Segmented_CCV2\annotations_3mst.csv",
    label_column = "mst_label"
)

Saved re-binned CSV → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations_3mst.csv
